# CCTV Shoplifting Detection — End-to-End Colab Notebook\n\nA computer vision portfolio project comparing two approaches to detecting shoplifting behavior in CCTV footage:\n\n1. **Classical baseline** — a non-deep-learning motion/Structural Similarity Index (SSIM) approach: watch a fixed Region of Interest on a shelf/display case, and flag any frame where it visibly changes from a reference. No training, no GPU, no labels required. This is a well-known OpenCV/computer-vision technique for simple change detection, not tied to any one specific article or paper.\n2. **YOLOv8 detector** — `yolov8n` fine-tuned into a 2-class detector (`shoplifting_person` / `not_shoplifting_person`) on a synthetic CCTV shoplifting dataset.\n3. **VLM captioning** — pairs each YOLO detection with the dataset's own ground-truth scene-description captions, for a qualitative view of what the model got right or wrong.\n\nThis notebook reproduces the full pipeline end-to-end — mount Drive, clone the repo, download/cache the dataset, run the EDA, run the classical baseline, train + evaluate + run the YOLO detector, demo the captioning module, and compare both detectors side by side — all from one place, meant to be run top-to-bottom via **Runtime → Run all**.\n\n**Dataset:** [simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm](https://www.kaggle.com/datasets/simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm) on Kaggle, created by **Simuletic**. Synthetic (simulated) CCTV footage with YOLO bounding-box/pose annotations and VLM-style scene captions. Used here for non-commercial research/portfolio purposes — see the repo README for the full dataset writeup and the sim-to-real limitations.\n\n**Tech stack:** Python, OpenCV, scikit-image (SSIM), Ultralytics YOLOv8, PyTorch, pandas / numpy / matplotlib, kagglehub.\n\n**Repo:** https://github.com/hrshimpi/cctv-shoplifting-detection\n\n**Author:** Himanshu Shimpi

In [ ]:
# Uninstall any pre-existing opencv build first to avoid two conflicting\n# cv2 packages, then install the headless build - Colab has no display,\n# so opencv-python-headless is correct here, not opencv-python.\n!pip uninstall -y opencv-python opencv-python-headless -q\n!pip install -q opencv-python-headless ultralytics scikit-image kagglehub pandas numpy matplotlib seaborn pyyaml tqdm\n\n!nvidia-smi

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
import os\n\nREPO_URL = \"https://github.com/hrshimpi/cctv-shoplifting-detection.git\"\nREPO_DIR = \"/content/cctv-shoplifting-detection\"\n\nif not os.path.exists(REPO_DIR):\n    !git clone {REPO_URL} {REPO_DIR}\nelse:\n    print(f\"{REPO_DIR} already exists - pulling latest instead of re-cloning.\")\n    !git -C {REPO_DIR} pull\n\n%cd {REPO_DIR}\n\nimport sys\nif REPO_DIR not in sys.path:\n    sys.path.insert(0, REPO_DIR)

## Dataset (Step 1)\n\nChecks for a cached copy of the dataset on Drive first, so re-running this notebook later skips re-downloading ~500MB from Kaggle every time. If nothing is cached yet, downloads it via `kagglehub` and caches a copy to Drive for next time.\n\nThis needs Kaggle API credentials. The safe way in Colab: add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Colab secrets** (key icon in the left sidebar) — never paste them into a cell as plain text. If this is a repeat run and the Drive cache already has the data, the credentials cell below can fail silently and it won't matter.

In [ ]:
import os\nfrom google.colab import userdata\n\ntry:\n    os.environ[\"KAGGLE_USERNAME\"] = userdata.get(\"KAGGLE_USERNAME\")\n    os.environ[\"KAGGLE_KEY\"] = userdata.get(\"KAGGLE_KEY\")\n    print(\"Kaggle credentials loaded from Colab secrets.\")\nexcept Exception:\n    print(\n        \"No KAGGLE_USERNAME/KAGGLE_KEY Colab secrets found. That's fine if the \"\n        \"Drive cache below already has the dataset - otherwise, add them as \"\n        \"Colab secrets (key icon, left sidebar) before running the next cell.\"\n    )

In [ ]:
import shutil\nfrom pathlib import Path\n\nimport kagglehub\n\nfrom src.data.download_dataset import DATASET_SLUG\n\nDRIVE_DATA_CACHE = Path(\"/content/drive/MyDrive/cctv_shoplifting_data\")\nREPO_DATA_DIR = Path(\"data\")\nREPO_DATA_DIR.mkdir(exist_ok=True)\n\n\ndef _copy_tree_contents(src: Path, dst: Path) -> None:\n    dst.mkdir(parents=True, exist_ok=True)\n    for item in src.iterdir():\n        dest_item = dst / item.name\n        if dest_item.exists():\n            continue\n        if item.is_dir():\n            shutil.copytree(item, dest_item)\n        else:\n            shutil.copy2(item, dest_item)\n\n\nif DRIVE_DATA_CACHE.exists() and any(DRIVE_DATA_CACHE.iterdir()):\n    print(f\"Found cached dataset on Drive at {DRIVE_DATA_CACHE} - copying into {REPO_DATA_DIR}/ ...\")\n    _copy_tree_contents(DRIVE_DATA_CACHE, REPO_DATA_DIR)\nelse:\n    print(f\"No cached copy at {DRIVE_DATA_CACHE} - downloading '{DATASET_SLUG}' via kagglehub ...\")\n    cache_path = Path(kagglehub.dataset_download(DATASET_SLUG))\n    _copy_tree_contents(cache_path, REPO_DATA_DIR)\n    print(f\"Caching a copy to {DRIVE_DATA_CACHE} for future runs ...\")\n    _copy_tree_contents(REPO_DATA_DIR, DRIVE_DATA_CACHE)\n\nprint(\"\\ndata/ now contains:\")\n!ls data/

## Exploratory Data Analysis (Step 1)\n\nRe-runs the same EDA script used in local development. It discovers the dataset's real structure at runtime rather than assuming one — this dataset ships no `data.yaml`/`classes.txt`, so the class signal comes from each source video's VLM-labels metadata instead (see the script's own docstring, or the repo README's \"Dataset\" section, for the full explanation) — then writes plots to `outputs/eda/`.

In [ ]:
!python src/data/eda.py\n\nfrom IPython.display import Image, display\n\nfor name in [\"class_distribution.png\", \"sample_grid.png\", \"resolution_histogram.png\"]:\n    print(f\"\\n{name}:\")\n    display(Image(filename=f\"outputs/eda/{name}\"))

## Classical baseline: SSIM / ROI motion detection (Step 2)\n\nWatches one fixed Region of Interest on a shelf/display table and flags any frame where it visibly changes from a reference, using the Structural Similarity Index on denoised grayscale crops. `motion_baseline.py` never calls `cv2.imshow` or `cv2.selectROI` — the cell below confirms that programmatically before running it, then previews a couple of frames with `google.colab.patches.cv2_imshow` (Colab's non-blocking substitute for `cv2.imshow`, since Colab has no display).

In [ ]:
import inspect\n\nfrom src.classical_cv import motion_baseline\n\n# Check for the actual call pattern (with the opening paren), not just the\n# bare name - the module's own docstring mentions \"cv2.imshow\" by name\n# while explaining that it's absent, which a plain substring check would\n# misfire on.\nsource_code = inspect.getsource(motion_baseline)\nassert \"cv2.imshow(\" not in source_code, \"motion_baseline.py must stay headless\"\nassert \"cv2.selectROI(\" not in source_code, \"motion_baseline.py must stay headless\"\nprint(\"Confirmed: no cv2.imshow() / cv2.selectROI() calls in motion_baseline.py.\")\n\ndetector = motion_baseline.run_on_video(\n    video_path=\"data/CCTV_Shoplifting_Dataset/videos/shoplifting1.mp4\",\n    roi=(175, 343, 75, 65),\n    threshold=0.8,\n    reference_mode=\"fixed\",\n    output_video_path=\"outputs/classical_baseline/colab_demo/annotated.mp4\",\n    alerts_dir=\"outputs/alerts/colab_demo\",\n    alerts_log_path=\"outputs/classical_baseline/colab_demo/alerts.jsonl\",\n)\nprint(\n    f\"\\nframes: {detector.frame_count}, alerts: {len(detector.alerts)}, \"\n    f\"avg similarity: {detector.average_similarity:.3f}\"\n)

In [ ]:
import cv2\nfrom google.colab.patches import cv2_imshow\n\ncap = cv2.VideoCapture(\"outputs/classical_baseline/colab_demo/annotated.mp4\")\nframes = []\nwhile True:\n    ok, frame = cap.read()\n    if not ok:\n        break\n    frames.append(frame)\ncap.release()\n\nprint(\"Frame 0 (quiet - box still on the table):\")\ncv2_imshow(frames[0])\nprint(\"Frame 60 (flagged - box gone):\")\ncv2_imshow(frames[60])

## YOLOv8 detector: data prep, training, evaluation, inference (Step 3)\n\nBuilds a plain 2-class YOLO detection dataset from the raw download (the raw labels are YOLO-*pose* format with a single \"person\" class — see `prepare_yolo_data.py`'s docstring for why the class remap is necessary), fine-tunes `yolov8n`, evaluates it, and runs inference on sample images.\n\nTraining output goes to a **Drive path** (`/content/drive/MyDrive/cctv_shoplifting_outputs/runs`) so the weights and logs survive a Colab disconnect — re-running this notebook later can skip straight to evaluation/inference against the already-trained weights instead of retraining from scratch.

In [ ]:
!python src/detection/prepare_yolo_data.py

In [ ]:
import os\n\nimport torch\n\nDRIVE_RUNS_DIR = \"/content/drive/MyDrive/cctv_shoplifting_outputs/runs\"\ndevice = \"0\" if torch.cuda.is_available() else \"cpu\"\nprint(f\"Training on device: {device}\")\n\nbest_weights_path = f\"{DRIVE_RUNS_DIR}/train/weights/best.pt\"\nif os.path.exists(best_weights_path):\n    print(f\"Found existing trained weights at {best_weights_path} - skipping retraining.\")\n    print(\"Delete that folder on Drive (or pass a different --name) to force a fresh run.\")\nelse:\n    !python src/detection/train_yolo.py --epochs 30 --imgsz 320 --batch 8 --device {device} --project \"{DRIVE_RUNS_DIR}\" --name train

In [ ]:
!python src/detection/evaluate_yolo.py \\\n    --weights \"{DRIVE_RUNS_DIR}/train/weights/best.pt\" \\\n    --train-run-dir \"{DRIVE_RUNS_DIR}/train\" \\\n    --project \"{DRIVE_RUNS_DIR}\" \\\n    --name eval\n\nfrom IPython.display import Image, display\n\ndisplay(Image(filename=f\"{DRIVE_RUNS_DIR}/eval/confusion_matrix.png\"))\ndisplay(Image(filename=f\"{DRIVE_RUNS_DIR}/eval/BoxPR_curve.png\"))\ndisplay(Image(filename=f\"{DRIVE_RUNS_DIR}/train/loss_curves.png\"))

In [ ]:
!python src/detection/infer.py \\\n    --weights \"{DRIVE_RUNS_DIR}/train/weights/best.pt\" \\\n    --source outputs/yolo_dataset/images/val \\\n    --project \"{DRIVE_RUNS_DIR}\" \\\n    --name infer\n\nimport glob\n\nfrom IPython.display import Image, display\n\nsample_outputs = sorted(glob.glob(f\"{DRIVE_RUNS_DIR}/infer/*.jpg\"))[:4]\nfor path in sample_outputs:\n    display(Image(filename=path))

## VLM captioning demo (Step 3)\n\nPairs each sample frame's YOLO detection with the dataset's own ground-truth caption for that exact frame. Step 1's EDA already found real captions shipped with this dataset (`VLM-labels/*.json`), so this loads and pairs them rather than running a separate pretrained captioning model.

In [ ]:
from src.detection.caption_module import build_demo, pick_sample_images\n\nsamples = pick_sample_images(n_per_class=3)\n_ = build_demo(\n    weights=f\"{DRIVE_RUNS_DIR}/train/weights/best.pt\",\n    image_paths=samples,\n    log_path=\"outputs/yolo_runs/caption_demo.jsonl\",\n)

## Classical vs. deep learning: side-by-side comparison\n\nRuns both detectors on the same real clip, frame by frame, and produces one combined side-by-side annotated video: classical SSIM/ROI on the left, fine-tuned YOLOv8 on the right.

In [ ]:
from src.utils.compare_pipeline import compare_on_video, print_summary_table\n\nsummary = compare_on_video(\n    video_path=\"data/CCTV_Shoplifting_Dataset/videos/shoplifting1.mp4\",\n    weights=f\"{DRIVE_RUNS_DIR}/train/weights/best.pt\",\n    output_video_path=\"outputs/compare_pipeline/shoplifting1_side_by_side.mp4\",\n)\nprint_summary_table([summary])\n\nfrom IPython.display import Video, display\n\ndisplay(Video(summary[\"output_video\"], embed=True, width=800))

## Results & discussion\n\n**Final YOLOv8 metrics** (30 epochs, `yolov8n`, imgsz=320 — the numbers below are from a real local run of this exact pipeline; your own run above will differ slightly run-to-run, and should be noticeably faster on a GPU):\n\n| Class | Images | Instances | Precision | Recall | mAP50 | mAP50-95 |\n|---|---|---|---|---|---|---|\n| all | 130 | 275 | 0.497 | 0.445 | 0.283 | 0.181 |\n| not_shoplifting_person | 49 | 111 | 0.393 | 0.441 | 0.263 | 0.189 |\n| shoplifting_person | 81 | 164 | 0.600 | 0.449 | 0.303 | 0.173 |\n\n**Classical vs. deep learning:**\n- The classical SSIM/ROI baseline needs zero training and correctly flags \"something in this exact spot changed\" — but has no idea *what* changed. It flags a returned item and a concealed item identically (see the repo README's \"Baseline\" section for the concrete before/after evidence).\n- The fine-tuned YOLOv8 detector reliably finds *people* (it flagged 145/145 frames in both sample clips in local testing) but is much less reliable at *classifying* which class a person belongs to — the confusion matrix above shows `shoplifting_person` correctly identified only 9/164 times in the run behind these numbers, most often confused with `not_shoplifting_person` or missed entirely.\n- Neither one alone is a trustworthy verdict. What each is actually good at is complementary — motion detection for \"something happened here,\" object detection for \"here's where the person is\" — which is the practical case for combining both rather than picking one.\n\n**Limitations & responsible use:**\n- **Synthetic-to-real domain gap.** This entire pipeline is trained and evaluated on synthetic, simulated CCTV footage (see the dataset citation at the top) — not real store surveillance. Camera noise, compression artifacts, lighting, and real shoplifting behavior all differ from this synthetic data, and neither detector here has been validated against real footage.\n- **Class imbalance & small data.** 456 total frames from only 8 source videos, with a 260/196 shoplifting/not-shoplifting split — not a large or diverse enough sample to expect broad generalization.\n- **Single fixed camera angle per demo.** The classical baseline's ROI, and ultimately this whole pipeline's training data, come from a handful of fixed camera setups; a different store layout, camera height, or angle would need new ROI calibration and likely hurt YOLO accuracy too.\n- **Not a deployment-ready shoplifting detector.** Given the above, any real-world use would need a human reviewing every flagged alert before any action is taken — nothing in this repo should be used to automatically accuse, detain, or penalize anyone based on a model output alone.

## Syncing results back to GitHub\n\nEverything meant to persist (trained weights, training logs, eval plots) already lives under `/content/drive/MyDrive/cctv_shoplifting_outputs/` on your Drive — nothing here evaporates when this Colab session ends.\n\nIf you want to commit any of these results (metrics, plots) back to the repo:\n\n- **Simplest (recommended):** open `/content/drive/MyDrive/cctv_shoplifting_outputs/` in Google Drive (or Drive for Desktop), copy whatever small files you want (e.g. a new `confusion_matrix.png` into `reports/yolo/`), and commit them from VS Code like any other change — no credentials needed inside Colab at all.\n- **From inside Colab instead:** add a GitHub Personal Access Token as a Colab secret (e.g. named `GITHUB_TOKEN`) and use it to build an authenticated remote URL at run time — **never paste the token itself into a cell**:\n\n  ```python\n  from google.colab import userdata\n  token = userdata.get('GITHUB_TOKEN')\n  !git remote set-url origin https://{token}@github.com/hrshimpi/cctv-shoplifting-detection.git\n  !git add reports/yolo && git commit -m \"Update eval artifacts from Colab run\" && git push\n  ```